# Assignment 1 return data

Create the stock return table from the local raw extracts:

- `daily_stock_returns`: stock-level daily returns, with delisting returns incorporated on the delisting date.

Index and Treasury pulls were dropped from this pipeline: neither is read by any downstream
modeling notebook (see README.md for the audit).

Output is written to `../data/interim`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim"

print(PROJECT_ROOT)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

D:\Documents\School\cfm301\project1


## Load raw daily stock and delisting data

`daily_df.parquet` already contains the CRSP daily stock data linked to Compustat via CCM.
`dlist_df.parquet` contains CRSP daily delisting returns from `crsp_a_stock.dsedelist`.

In [2]:
daily_cols = [
    "date",
    "permno",
    "permco",
    "cusip",
    "ccm_gvkey",
    "ret",
    "retx",
    "prc",
    "shrout",
    "cfacpr",
    "cfacshr",
    "name_exchcd",
    "name_shrcd",
    "ccm_linktype",
    "ccm_linkprim",
]

dlist_cols = [
    "permno",
    "dlstdt",
    "dlstcd",
    "dlret",
    "dlretx",
    "dlprc",
    "dlamt",
    "nextdt",
    "dlpdt",
    "nwperm",
    "nwcomp",
    "acperm",
    "accomp",
]

daily_df = pd.read_parquet(RAW_DATA_DIR / "daily_df.parquet", columns=daily_cols)
dlist_df = pd.read_parquet(RAW_DATA_DIR / "dlist_df.parquet", columns=dlist_cols)

daily_df["date"] = pd.to_datetime(daily_df["date"])
dlist_df["dlstdt"] = pd.to_datetime(dlist_df["dlstdt"])

## Delisting return methodology

Delisting returns belong in the stock-level daily return table, not the index table.

CRSP's `dlstdt` does not always coincide with an existing `dsf` trading date for that permno -- trading can stop days or weeks before the delisting date is recorded. A plain merge on `(permno, date)` silently drops any delisting return whose `dlstdt` has no matching row. To keep every delisting return, split into two cases:

1. **`dlstdt` matches an existing trading date** -- compound the ordinary CRSP daily return with the delisting return: `(1 + ret) * (1 + dlret) - 1`. If `ret` is missing that day but `dlret` exists, use `dlret` as the terminal return. Same logic for `retx`/`dlretx`.
2. **`dlstdt` has no matching trading date** (stock stopped trading before it was formally delisted) -- append a new row dated `dlstdt` for that permno, carrying forward its last known identifying fields (permco, cusip, gvkey, exchange/share code, link fields), with `ret`/`retx` left null and `ret_including_delist`/`retx_including_delist` set directly to `dlret`/`dlretx`.

Permnos that never appear in `daily_df` at all (filtered out upstream by the exchange/share-code universe filters) are left out, same as before.

In [3]:
daily_stock_returns = daily_df.merge(
    dlist_df,
    how="left",
    left_on=["permno", "date"],
    right_on=["permno", "dlstdt"],
    validate="many_to_one",
)

# Delisting events for permnos that are in the sample but whose dlstdt never
# lines up with an existing trading date -- these would otherwise be dropped
# silently by the merge above.
permnos_in_sample = daily_df["permno"].unique()
matched_dlstdt = set(
    zip(
        daily_stock_returns.loc[daily_stock_returns["dlstdt"].notna(), "permno"],
        daily_stock_returns.loc[daily_stock_returns["dlstdt"].notna(), "dlstdt"],
    )
)

dlist_in_sample = dlist_df[dlist_df["permno"].isin(permnos_in_sample)]
is_unmatched = ~dlist_in_sample.apply(
    lambda row: (row["permno"], row["dlstdt"]) in matched_dlstdt, axis=1
)
unmatched_dlist = dlist_in_sample[is_unmatched]

last_daily_row = (
    daily_df.sort_values(["permno", "date"])
    .groupby("permno", as_index=False)
    .last()
    .drop(columns=["date", "ret", "retx", "prc"])
)

appended_rows = unmatched_dlist.merge(last_daily_row, on="permno", how="left")
appended_rows["date"] = appended_rows["dlstdt"]
appended_rows["ret"] = np.nan
appended_rows["retx"] = np.nan
appended_rows["prc"] = np.nan

daily_stock_returns = pd.concat(
    [daily_stock_returns, appended_rows], ignore_index=True, sort=False
)

ret = pd.to_numeric(daily_stock_returns["ret"], errors="coerce")
retx = pd.to_numeric(daily_stock_returns["retx"], errors="coerce")
dlret = pd.to_numeric(daily_stock_returns["dlret"], errors="coerce")
dlretx = pd.to_numeric(daily_stock_returns["dlretx"], errors="coerce")

daily_stock_returns["ret_including_delist"] = np.where(
    dlret.notna(),
    (1.0 + ret.fillna(0.0)) * (1.0 + dlret) - 1.0,
    ret,
)

daily_stock_returns["retx_including_delist"] = np.where(
    dlretx.notna(),
    (1.0 + retx.fillna(0.0)) * (1.0 + dlretx) - 1.0,
    retx,
)

daily_stock_returns = daily_stock_returns.sort_values(
    ["permno", "date"], kind="stable"
).reset_index(drop=True)

## Rename columns for readability

The raw CRSP names are compact but hard to read. This output table uses descriptive names so
teammates can understand the fields without constantly checking CRSP documentation.

In [4]:
stock_column_names = {
    "date": "trade_date",
    "permno": "crsp_security_id_permno",
    "permco": "crsp_company_id_permco",
    "cusip": "cusip",
    "ccm_gvkey": "compustat_gvkey",
    "ret": "daily_return",
    "retx": "daily_return_excluding_distributions",
    "ret_including_delist": "daily_return_with_delisting",
    "retx_including_delist": "daily_return_excluding_distributions_with_delisting",
    "prc": "price",
    "shrout": "shares_outstanding_thousands",
    "cfacpr": "price_adjustment_factor",
    "cfacshr": "share_adjustment_factor",
    "name_exchcd": "exchange_code",
    "name_shrcd": "share_code",
    "ccm_linktype": "ccm_link_type",
    "ccm_linkprim": "ccm_primary_link_flag",
    "dlstdt": "delisting_date",
    "dlstcd": "delisting_code",
    "dlret": "delisting_return",
    "dlretx": "delisting_return_excluding_distributions",
    "dlprc": "delisting_price",
    "dlamt": "delisting_payment_amount",
    "nextdt": "next_distribution_date",
    "dlpdt": "delisting_payment_date",
    "nwperm": "new_crsp_security_id_permno",
    "nwcomp": "new_crsp_company_id_permco",
    "acperm": "acquiring_crsp_security_id_permno",
    "accomp": "acquiring_crsp_company_id_permco",
}

daily_stock_returns = daily_stock_returns.rename(columns=stock_column_names)

## Save assignment outputs

In [5]:
stock_output_path = OUTPUT_DIR / "daily_stock_returns.parquet"

daily_stock_returns.to_parquet(stock_output_path, index=False)

stock_output_path

WindowsPath('C:/Users/bigna/Desktop/Quant Projects/Factor Models/claude_final_submission/data/interim/daily_stock_returns.parquet')

## Quick checks

In [6]:
summary = {
    "stock_rows": len(daily_stock_returns),
    "stock_permnos": daily_stock_returns["crsp_security_id_permno"].nunique(),
    "stock_min_date": daily_stock_returns["trade_date"].min(),
    "stock_max_date": daily_stock_returns["trade_date"].max(),
    "delisting_rows_matched": daily_stock_returns["delisting_return"].notna().sum(),
}

summary

{'stock_rows': 8126800,
 'stock_permnos': 6832,
 'stock_min_date': Timestamp('2003-01-02 00:00:00'),
 'stock_max_date': Timestamp('2009-12-31 00:00:00'),
 'delisting_rows_matched': np.int64(2633)}